Knowledge Editing

In [3]:
%%bash
!(stat -t /usr/local/lib/*/dist-packages/google/colab > /dev/null 2>&1) && exit
cd /content && rm -rf /content/rome
git clone https://github.com/kmeng01/rome rome > install.log 2>&1
pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
pip install --upgrade google-cloud-storage >> install.log 2>&1

In [5]:
!pip install torch transformers transformer-lens datasets einops accelerate nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.4 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=21e1314fced6c7c1622b07852c52f3a3f04915cf147d9476a3f89085ffa6549e
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


Download Model

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

In [8]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model.to(device)
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1600)
    (wpe): Embedding(1024, 1600)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-47): 48 x GPT2Block(
        (ln_1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=4800, nx=1600)
          (c_proj): Conv1D(nf=1600, nx=1600)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=6400, nx=1600)
          (c_proj): Conv1D(nf=1600, nx=6400)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1600, out_features=50257, bias=False)
)

In [9]:
prompt = "The Eiffel Tower is located in the city of"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
  out = model.generate(**inputs, max_new_tokens=10)
print(tokenizer.decode(out[0], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The Eiffel Tower is located in the city of Paris, France. The tower is the tallest structure


In [12]:
print(out[0, -1])

tensor(4645)


Finding the exact probability

In [13]:
with torch.no_grad():
  logits = model(**inputs).logits
  print(logits[0, -1])
  probs = F.softmax(logits[0, -1], dim=-1)

paris_id = tokenizer.encode(" Paris", add_special_tokens=False)[0]
print(f"P(Paris) before edit: {probs[paris_id].item():.4f}")

tensor([ -0.3662,   0.6314,  -3.3538,  ..., -10.4531,  -6.4084,   0.2281])
P(Paris) before edit: 0.9341


Set up Subject && target tokens

In [32]:
subject = " Eiffel Tower"
target = " Paris"

In [33]:
full_tokens = inputs['input_ids'][0].tolist()
subject_ids = tokenizer.encode(subject, add_special_tokens=False)
subject_ids

[412, 733, 417, 8765]

In [34]:
def find_sublist(lst, sub):
  for i in range(len(lst) - len(sub) + 1):
    if lst[i:i+len(sub)] == sub:
      return list(range(i, i+len(sub)))
  return []

In [35]:
subject_positions = find_sublist(full_tokens, subject_ids)
token_labels = tokenizer.convert_ids_to_tokens(full_tokens)

In [36]:
print(f"Tokens: {token_labels}")
print(f"Subject positions: {subject_positions}")

Tokens: ['The', 'ĠE', 'iff', 'el', 'ĠTower', 'Ġis', 'Ġlocated', 'Ġin', 'Ġthe', 'Ġcity', 'Ġof']
Subject positions: [1, 2, 3, 4]
